# <img style="float: left; padding-right: 10px; width: 100px" src="https://raw.githubusercontent.com/sbildea/SEED/main/imagini/unitbv_mie.png"> <center>Știința Explorării și Exploatării Datelor - SEED1</center>


##    <center>Curs 6: Evaluare și optimizare: când este un model „bun”?</center>
**Universitatea Transilvania**<br>
**SIIPA, Semestrul I**<br>
**Instructor:** Conf. dr. Teodor &Scedil;tefan B&icirc;ldea<br>
**Contact:** teodor.bildea@unitbv.ro<br>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sbildea/SEED/blob/main/cursuri/06-evaluare-optimizare/curs-06-evaluare-optimizare.ipynb)

In [ ]:
# Configurare curs SEED: rulați această celulă prima.
# Colab: descarcă materialele cursului și instalează pachetele lipsă (aprox. un minut).
# Local: găsește singură rădăcina repository-ului, oricare ar fi folderul curent al kernelului.
import os, sys
MODUL = "06-evaluare-optimizare"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SEED"):
        !git clone -q --depth 1 https://github.com/sbildea/SEED.git /content/SEED
        !pip install -q -r /content/SEED/requirements.txt
    RADACINA = "/content/SEED"
else:
    RADACINA = os.path.abspath(".")
    while not os.path.isdir(os.path.join(RADACINA, "seed")) and os.path.dirname(RADACINA) != RADACINA:
        RADACINA = os.path.dirname(RADACINA)
    import importlib.util, socket
    lipsa = [p for p in ("pandas", "matplotlib", "seaborn", "sklearn") if importlib.util.find_spec(p) is None]
    if lipsa:
        print("Lipsesc pachetele", lipsa, "- în terminal: pip install -r requirements-local.txt")
    try:
        socket.create_connection(("raw.githubusercontent.com", 443), 2).close()
    except OSError:
        print("Fără internet: imaginile din text nu se văd. O singură dată, în terminal: python tools/imagini_offline.py")
os.chdir(os.path.join(RADACINA, "cursuri", MODUL))
sys.path.insert(0, RADACINA)

## Problema de azi

Un operator de telecomunicații pierde în fiecare lună clienți. Managerul de retenție ne spune:

> *„Vreau o listă cu clienții care urmează să plece, ca să le oferim o reducere. Fiecare ofertă ne costă bani, deci nu o putem trimite tuturor. Cât de bun trebuie să fie modelul? Și de unde știu că va funcționa și luna viitoare?”*

Acestea sunt exact întrebările acestui curs:

1. **Generalizare:** cum estimăm cât de bine va merge modelul pe clienți pe care nu i-a văzut?
2. **Validare încrucișată:** de ce nu ne bazăm pe o singură împărțire a datelor.
3. **Metrica vine din problemă:** ce greșeli costă mai mult și cum transformăm matricea de confuzie în bani.
4. **Hiperparametri și optimizare:** cum alegem setările modelului fără să „trișăm”.
5. **Data leakage:** de ce un model „prea bun ca să fie adevărat” chiar nu este adevărat.

Codul este scurt și servește ca ilustrație: important este conceptul. La seminar veți obține astfel de rezultate cerându-le unui asistent AI (vezi [ghidul AI](https://github.com/sbildea/SEED/blob/main/docs/ghid-programare-asistata-ai.md)).

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from seed.date import incarca

df = incarca("lab-set1")[0]                       # Telco Customer Churn
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
y = (df["Churn"] == "Yes").astype(int)            # 1 = clientul pleacă
X = df.drop(columns=["customerID", "Churn"])
print(X.shape, "| proporția clienților care pleacă:", round(y.mean(), 3))

(7043, 19) | proporția clienților care pleacă: 0.265


## 1. Generalizare: modelul lucrează pe clienți noi

Un model nu ne interesează pentru clienții din trecut (știm deja ce au făcut), ci pentru cei de **luna viitoare**. Ca la un examen: dacă subiectele sunt exact exercițiile de la seminar, nota nu arată dacă ați înțeles, ci dacă ați memorat.

De aceea împărțim datele:

| Set | Rol | Cine îl „vede” |
|---|---|---|
| **antrenare** | modelul învață din el | modelul |
| **validare** (sau validare încrucișată) | alegem modelul și setările lui | noi, de mai multe ori |
| **test** | estimarea finală, „examenul” | **o singură dată**, la final |

Comparăm un model simplu (regresie logistică) cu un arbore de decizie lăsat să crească fără limită.

### Linii de procesare (pipeline)

Un proiect de învățare automată înlănțuie pași. Cei mai mulți **transformă datele** (filtrare, imputare, codificare, standardizare, discretizare, atribute derivate), iar **ultimul** antrenează modelul. Un `Pipeline` leagă toți pașii într-un singur obiect: *date inițiale → transformator 1 → … → transformator n → model*.

De ce contează la evaluare: în validarea încrucișată, `cross_val_score` reînvață transformările **din nou pe fiecare fold, doar pe partea de antrenament**. Dacă ați scala datele înainte de validare, fiecare fold ar „vedea” statistici din propriul set de test (*data leakage*). Același obiect se salvează și se aplică ulterior pe date noi, cu aceiași pași. Celula de mai jos construiește exact o astfel de linie: pregătire (codificare + standardizare) și model.

Slide-uri de referință: [extra-06-pipeline-uri.pdf](extra-06-pipeline-uri.pdf).

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# setul de test se pune deoparte PRIMUL și nu se mai atinge până la final
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=17)

def pregatire():
    # scalare pentru coloanele numerice, codificare one-hot pentru cele categoriale
    return make_column_transformer(
        (StandardScaler(), make_column_selector(dtype_include="number")),
        (OneHotEncoder(handle_unknown="ignore"), make_column_selector(dtype_exclude="number")))

modele = {
    "regresie logistică": make_pipeline(pregatire(), LogisticRegression(max_iter=1000)),
    "arbore fără limită": make_pipeline(pregatire(), DecisionTreeClassifier(random_state=17)),
}
for nume, model in modele.items():
    model.fit(X_train, y_train)
    print(f"{nume:20s} acuratețe pe antrenare {model.score(X_train, y_train):.3f} | pe test {model.score(X_test, y_test):.3f}")

regresie logistică   acuratețe pe antrenare 0.804 | pe test 0.813
arbore fără limită   acuratețe pe antrenare 0.998 | pe test 0.736


Arborele are acuratețe aproape perfectă pe antrenare și mult mai slabă pe test: a **memorat** clienții, nu a învățat reguli generale (**supraînvățare**, *overfitting*).

Dar atenție și la acuratețe: aproximativ 73% dintre clienți rămân. Un „model” care spune mereu *„nimeni nu pleacă”* are deja această acuratețe, fără să fie de vreun folos managerului.

In [4]:
from sklearn.dummy import DummyClassifier

naiv = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"modelul „nimeni nu pleacă”: acuratețe pe test {naiv.score(X_test, y_test):.3f}, clienți care pleacă găsiți: 0")

modelul „nimeni nu pleacă”: acuratețe pe test 0.735, clienți care pleacă găsiți: 0


**Concluzie:** comparăm întotdeauna cu un **model de referință** naiv, iar acuratețea singură poate înșela când clasele sunt dezechilibrate.

## 2. Validarea încrucișată: o estimare stabilă

Dacă împărțim altfel datele (alt `random_state`), scorul se schimbă. Care este cel „adevărat”? Validarea încrucișată (*cross-validation*) împarte antrenarea în 5 părți și antrenează de 5 ori, de fiecare dată ținând deoparte altă parte:

| Tura | Partea 1 | Partea 2 | Partea 3 | Partea 4 | Partea 5 |
|---|---|---|---|---|---|
| 1 | **validare** | antrenare | antrenare | antrenare | antrenare |
| 2 | antrenare | **validare** | antrenare | antrenare | antrenare |
| … | | | | | |
| 5 | antrenare | antrenare | antrenare | antrenare | **validare** |

Raportăm **media** celor 5 scoruri (și cât variază). Fiecare client este folosit o dată pentru validare. Setul de test rămâne în continuare neatins.

In [5]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

model = modele["regresie logistică"]

# 10 împărțiri simple, diferite doar prin random_state
scoruri_simple = []
for seed in range(10):
    Xa, Xv, ya, yv = train_test_split(X_train, y_train, test_size=0.25, stratify=y_train, random_state=seed)
    scoruri_simple.append(model.fit(Xa, ya).score(Xv, yv))
print(f"o singură împărțire: acuratețe între {min(scoruri_simple):.3f} și {max(scoruri_simple):.3f}, după cum „pică” datele")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=17)
scoruri_cv = cross_val_score(model, X_train, y_train, cv=cv)
print(f"validare încrucișată (5 ture): {scoruri_cv.mean():.3f} ± {scoruri_cv.std():.3f}")

o singură împărțire: acuratețe între 0.795 și 0.812, după cum „pică” datele
validare încrucișată (5 ture): 0.802 ± 0.013


## 3. Metrica vine din problemă: matricea de confuzie în bani

Modelul dă pentru fiecare client o **probabilitate** de plecare. Noi alegem **pragul** peste care îi trimitem oferta. Fiecare decizie are o consecință:

| | clientul ar fi plecat | clientul ar fi rămas |
|---|---|---|
| **primește oferta** | adevărat pozitiv (TP): îl păstrăm cu o anumită șansă | fals pozitiv (FP): am cheltuit o ofertă degeaba |
| **nu primește oferta** | fals negativ (FN): îl pierdem | adevărat negativ (TN): nimic de făcut |

Cifrele le dă **clientul nostru**, nu statistica. Să presupunem: o ofertă costă **50 lei**; un client păstrat aduce **600 lei** pe an; oferta convinge **jumătate** dintre cei care voiau să plece.

- câștig pentru un TP: 0,5 × 600 − 50 = **250 lei**
- cost pentru un FP: **−50 lei**
- FN și TN: 0 (față de situația în care nu facem nimic)

Alegem pragul care maximizează profitul. Ca să nu „trișăm”, îl alegem pe **predicțiile din validarea încrucișată** pe setul de antrenare, nu pe setul de test.

In [6]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix

CASTIG_TP, COST_FP = 250, -50

def profit(y_adevarat, probabilitati, prag):
    tn, fp, fn, tp = confusion_matrix(y_adevarat, (probabilitati >= prag).astype(int)).ravel()
    return tp * CASTIG_TP + fp * COST_FP

p_cv = cross_val_predict(model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
praguri = np.round(np.arange(0.05, 0.96, 0.05), 2)
profituri = [profit(y_train, p_cv, prag) for prag in praguri]
prag_optim = praguri[int(np.argmax(profituri))]

plt.figure(figsize=(8, 4))
plt.plot(praguri, np.array(profituri) / 1000, marker="o")
plt.axvline(0.5, color="grey", linestyle="--", label="pragul implicit 0,5")
plt.axvline(prag_optim, color="green", label=f"pragul cu profit maxim {prag_optim}")
plt.xlabel("pragul de probabilitate peste care trimitem oferta")
plt.ylabel("profit estimat (mii lei)")
plt.title("Profitul campaniei în funcție de prag (validare încrucișată)")
plt.legend();

Pragul cu profit maxim este **sub 0,5**: un client pierdut costă mult mai mult decât o ofertă trimisă degeaba, deci merită să trimitem mai multe oferte. **Pragul este o decizie de business, nu o setare implicită.** Dacă oferta ar fi scumpă, pragul optim ar crește.

Abia acum folosim setul de test, **o singură dată**, ca să vedem ce ar fi însemnat cele două praguri pe clienți noi.

In [7]:
from sklearn.metrics import precision_score, recall_score

model.fit(X_train, y_train)
p_test = model.predict_proba(X_test)[:, 1]
rezultate = []
for nume, prag in [("prag implicit 0,5", 0.5), (f"prag optim {prag_optim}", prag_optim)]:
    pred = (p_test >= prag).astype(int)
    rezultate.append({"decizie": nume, "oferte trimise": int(pred.sum()),
                      "precision": round(precision_score(y_test, pred), 2),
                      "recall": round(recall_score(y_test, pred), 2),
                      "profit pe test (lei)": profit(y_test, p_test, prag)})
pd.DataFrame(rezultate).set_index("decizie")

,oferte trimise,precision,recall,profit pe test (lei)
decizie,,,,
"prag implicit 0,5",312,0.68,0.56,47700
prag optim 0.15,780,0.44,0.93,64800


**Precision** = din clienții care primesc oferta, câți chiar voiau să plece. **Recall** = din clienții care voiau să plece, pe câți i-am găsit. Pragul mai mic crește recall-ul și scade precision-ul; profitul ne spune care compromis este cel bun **pentru această problemă**.

## 4. Ce metrică alegem: un rezumat

Regula: **alegeți metrica înainte de a antrena modele**, pornind de la costul greșelilor.

| Metrica | Răspunde la întrebarea | Când o alegem |
|---|---|---|
| acuratețe | ce procent de decizii sunt corecte? | clase echilibrate, greșeli cu cost similar |
| precision | când spun „da”, cât de des am dreptate? | o alarmă falsă costă mult (ex.: blocăm un card bancar) |
| recall | din cazurile reale, câte găsesc? | un caz ratat costă mult (ex.: boală, fraudă, client pierdut) |
| F1 | un echilibru între precision și recall | clase dezechilibrate, fără costuri cunoscute |
| ROC-AUC | cât de bine ordonează modelul cazurile după risc? | comparăm modele independent de prag |
| MAE / RMSE | cât greșesc, în medie, în unitățile țintei? | regresie (prețuri, vânzări, cereri) |
| **profit / cost** | cât câștigă sau pierde afacerea? | **când clientul ne poate da cifrele** |

## 5. Hiperparametri și optimizare

- **Parametrii** îi învață modelul din date (de exemplu coeficienții regresiei).
- **Hiperparametrii** îi alegem noi înainte de antrenare (de exemplu adâncimea maximă a unui arbore).

Hiperparametrii controlează compromisul dintre un model prea simplu și unul care memorează. Graficul de mai jos arată arborele de decizie pentru adâncimi de la 1 la 15: scorul pe antrenare crește mereu, scorul din validarea încrucișată crește, apoi scade.

In [8]:
from sklearn.model_selection import validation_curve

adancimi = list(range(1, 16))
scor_antr, scor_val = validation_curve(
    make_pipeline(pregatire(), DecisionTreeClassifier(random_state=17)), X_train, y_train,
    param_name="decisiontreeclassifier__max_depth", param_range=adancimi, cv=cv, scoring="roc_auc")

plt.figure(figsize=(8, 4))
plt.plot(adancimi, scor_antr.mean(axis=1), marker="o", label="antrenare")
plt.plot(adancimi, scor_val.mean(axis=1), marker="o", label="validare încrucișată")
plt.xlabel("adâncimea maximă a arborelui (hiperparametru)")
plt.ylabel("ROC-AUC")
plt.title("Prea simplu ← → memorează")
plt.legend();
print("adâncimea cu cel mai bun scor în validare:", adancimi[int(np.argmax(scor_val.mean(axis=1)))])

adâncimea cu cel mai bun scor în validare: 5


Cu un singur hiperparametru putem încerca toate valorile. Modelele reale au însă mai mulți (adâncime, număr de arbori, rată de învățare…) și combinațiile explodează. **Optuna** caută inteligent: învață din încercările anterioare unde merită să caute, într-un **buget de timp** fixat de noi. Ce îi cerem să optimizeze (funcția obiectiv) este **media din validarea încrucișată a metricii alese la pasul 4**. La seminar vom face exact asta.

## 6. Data leakage: când modelul „vede viitorul”

**Data leakage** (scurgerea de informație) apare când modelul primește la antrenare o informație pe care **nu o va avea** în momentul în care trebuie să decidă. Exemple din sisteme ERP / CRM:

- data cererii de reziliere, sau numărul de apeluri la departamentul de reziliere (apar *după* decizia clientului);
- un câmp „motiv închidere cont” completat abia la plecare;
- scalarea sau completarea valorilor lipsă calculată pe **tot** setul, inclusiv pe test.

Simulăm o astfel de coloană: „a depus cerere de reziliere”.

In [9]:
rng = np.random.default_rng(17)
X_cu_viitor = X_train.copy()
# coloană „din viitor”: în CRM apare abia după ce clientul a decis să plece
X_cu_viitor["CerereReziliere"] = np.where(y_train == 1, rng.random(len(y_train)) < 0.9,
                                          rng.random(len(y_train)) < 0.02).astype(int)

for nume, date in [("date corecte", X_train), ("cu coloana din viitor", X_cu_viitor)]:
    scor = cross_val_score(make_pipeline(pregatire(), LogisticRegression(max_iter=1000)),
                           date, y_train, cv=cv, scoring="roc_auc").mean()
    print(f"{nume:22s} ROC-AUC {scor:.3f}")

date corecte           ROC-AUC 0.843


cu coloana din viitor  ROC-AUC 0.979


Un scor aproape perfect este un **semnal de alarmă**, nu un motiv de bucurie. Întrebați mereu: *„Aș avea această coloană în momentul în care trebuie să iau decizia?”*

**Lista de verificare împotriva scurgerilor:**

- [ ] setul de test a fost separat primul și folosit o singură dată;
- [ ] scalarea, codificarea și completarea valorilor lipsă sunt în `pipeline` (învățate doar pe antrenare);
- [ ] fiecare coloană există în momentul deciziei (fără informații „din viitor”);
- [ ] pragul și hiperparametrii au fost aleși pe validare, nu pe test.

## 7. Ce îi spunem managerului

Un consultant nu predă un ROC-AUC, ci o recomandare cu cifre. Celula de mai jos scrie un rezumat folosind **doar** rezultatele calculate mai sus.

In [10]:
optim = rezultate[1]
implicit = rezultate[0]
lei = lambda x: f"{x:,}".replace(",", ".")   # 64.800, ca în română
print(f"""Rezumat pentru managerul de retenție
- Pe {len(y_test)} clienți pe care modelul nu i-a văzut, recomandăm trimiterea ofertei către {optim['oferte trimise']} clienți
  (cei cu probabilitate de plecare de cel puțin {prag_optim}).
- Astfel găsim {optim['recall']:.0%} dintre clienții care urmau să plece; {optim['precision']:.0%} dintre ofertele trimise ajung la clienți care chiar voiau să plece.
- Profit estimat: {lei(optim['profit pe test (lei)'])} lei, față de {lei(implicit['profit pe test (lei)'])} lei cu pragul implicit de 0,5.
- Ipoteze: ofertă 50 lei, client păstrat 600 lei/an, oferta convinge 1 din 2. Dacă aceste cifre se schimbă, pragul se recalculează.
- Limită: modelul învață din comportamentul trecut; trebuie reverificat lunar pe date noi.""")

Rezumat pentru managerul de retenție
- Pe 1409 clienți pe care modelul nu i-a văzut, recomandăm trimiterea ofertei către 780 clienți
  (cei cu probabilitate de plecare de cel puțin 0.15).
- Astfel găsim 93% dintre clienții care urmau să plece; 44% dintre ofertele trimise ajung la clienți care chiar voiau să plece.
- Profit estimat: 64.800 lei, față de 47.700 lei cu pragul implicit de 0,5.
- Ipoteze: ofertă 50 lei, client păstrat 600 lei/an, oferta convinge 1 din 2. Dacă aceste cifre se schimbă, pragul se recalculează.
- Limită: modelul învață din comportamentul trecut; trebuie reverificat lunar pe date noi.


## Puntea curs → seminar

**Concepte cheie**

1. **Metrica vine din problemă:** recall, precision, F1, ROC-AUC sau RMSE, după costul greșelilor.
2. **Validarea încrucișată:** o estimare stabilă, în locul unei singure împărțiri norocoase.
3. **Data leakage:** setul de test se folosește o singură dată; transformările stau în pipeline.
4. **Optimizare cu buget:** Optuna caută hiperparametrii mai eficient decât grid search.

**Ce ceri asistentului AI** (exemplu de cerere)

> Folosind Optuna cu validare încrucișată stratificată în 5 ture, optimizează recall pentru prezicerea `AHD` din `Heart.csv`, comparând regresia logistică și random forest, cu buget de 60 de secunde și `random_state=17`. Pune deoparte 20% date de test înainte și evaluează pe ele o singură dată, cu matrice de confuzie.

**Ce verifici înainte să accepți rezultatul**

- [ ] setul de test a fost separat înainte de Optuna și nu apare în funcția obiectiv;
- [ ] transformările sunt în pipeline sau învățate doar pe antrenare;
- [ ] metrica optimizată este cea argumentată din problemă;
- [ ] poți explica diferența dintre scorul CV și scorul de test.

Continuăm la seminar: [seminar-06-validare-incrucisata-optuna.ipynb](seminar-06-validare-incrucisata-optuna.ipynb).